In [1]:
"""
Примеры кода "Структурированный вывод и Output Parsers"

Этот модуль демонстрирует:
1. Базовые парсеры (StrOutputParser, JsonOutputParser)
2. PydanticOutputParser и типизация
3. with_structured_output для гарантированного JSON
4. Обработка ошибок парсинга
5. Продвинутые паттерны (вложенные схемы, Enum, валидаторы)
6. Практические рецепты
"""

'\nПримеры кода "Структурированный вывод и Output Parsers"\n\nЭтот модуль демонстрирует:\n1. Базовые парсеры (StrOutputParser, JsonOutputParser)\n2. PydanticOutputParser и типизация\n3. with_structured_output для гарантированного JSON\n4. Обработка ошибок парсинга\n5. Продвинутые паттерны (вложенные схемы, Enum, валидаторы)\n6. Практические рецепты\n'

In [2]:
import os
from typing import List, Dict, Any, Optional, Union
from enum import Enum
from datetime import datetime
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

# ============================================================================
# ЧАСТЬ 1: БАЗОВЫЕ ПАРСЕРЫ
# ============================================================================

In [ ]:
from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    CommaSeparatedListOutputParser,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


In [6]:
"""
StrOutputParser — извлечение текста из ответа модели.
"""
print("=" * 60)
print("ПРИМЕР 1: StrOutputParser")
print("=" * 60)

model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)
parser = StrOutputParser()

# Без парсера — получаем AIMessage
print("\n1.1. Без парсера:")
raw_response = model.invoke("Скажи привет")
print(f"  Тип: {type(raw_response).__name__}")
print(f"  Содержимое: {raw_response}")

# С парсером — получаем строку
print("\n1.2. С парсером:")
chain = model | parser
parsed_response = chain.invoke("Скажи привет")
print(f"  Тип: {type(parsed_response).__name__}")
print(f"  Содержимое: {parsed_response}")

ПРИМЕР 1: StrOutputParser

1.1. Без парсера:
  Тип: AIMessage
  Содержимое: content='Привет! Как я могу помочь тебе сегодня?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 11, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 8.25e-06, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 8.25e-06, 'upstream_inference_prompt_cost': 1.65e-06, 'upstream_inference_completions_cost': 6.6e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'gen-1770604062-O5Ozz1KYTXW8LNppI7X8', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019c4039-e432-7cf3-9d0b-fd93825da9d5-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 11, 'output_tokens': 11, 'total

In [7]:
"""
JsonOutputParser — извлечение JSON из ответа.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 2: JsonOutputParser")
print("=" * 60)

model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)
parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_template(
    """Верни информацию о {animal} в формате JSON с полями:
    - name: название на русском
    - type: тип животного
    - habitat: место обитания

    Верни ТОЛЬКО JSON, без пояснений."""
)

chain = prompt | model | parser

print("\n2.1. Извлечение JSON:")
result = chain.invoke({"animal": "пингвин"})
print(f"  Тип результата: {type(result).__name__}")
print(f"  Результат: {result}")

# Парсер умеет извлекать JSON из markdown
print("\n2.2. Парсер извлекает JSON из markdown блоков:")
raw_with_markdown = '```json\n{"name": "Python", "version": "3.12"}\n```'
parsed = parser.parse(raw_with_markdown)
print(f"  Вход: {raw_with_markdown}")
print(f"  Выход: {parsed}")


ПРИМЕР 2: JsonOutputParser

2.1. Извлечение JSON:
  Тип результата: dict
  Результат: {'name': 'Пингвин', 'type': 'птица', 'habitat': 'Антарктика и субантарктические острова'}

2.2. Парсер извлекает JSON из markdown блоков:
  Вход: ```json
{"name": "Python", "version": "3.12"}
```
  Выход: {'name': 'Python', 'version': '3.12'}


In [8]:
"""
CommaSeparatedListOutputParser — парсинг списков.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 3: CommaSeparatedListOutputParser")
print("=" * 60)

model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)
parser = CommaSeparatedListOutputParser()

prompt = ChatPromptTemplate.from_template(
    """Назови 5 языков программирования через запятую.
    {format_instructions}"""
)

# Парсер может предоставить инструкции для промпта
chain = prompt.partial(format_instructions=parser.get_format_instructions()) | model | parser

print("\n3.1. Парсинг списка:")
result = chain.invoke({})
print(f"  Тип: {type(result).__name__}")
print(f"  Результат: {result}")
print(f"  Длина списка: {len(result)}")


ПРИМЕР 3: CommaSeparatedListOutputParser

3.1. Парсинг списка:
  Тип: list
  Результат: ['Python', 'Java', 'C++', 'JavaScript', 'Ruby']
  Длина списка: 5


# ============================================================================
# ЧАСТЬ 2: PYDANTICOUTPUTPARSER
# ============================================================================


In [ ]:
"""
PydanticOutputParser с простой схемой.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 4: PydanticOutputParser (базовый)")
print("=" * 60)

from langchain_core.output_parsers import PydanticOutputParser


# Определяем схему данных
class Person(BaseModel):
    name: str = Field(description="Имя человека")
    age: int = Field(description="Возраст в годах")
    occupation: str = Field(description="Профессия")


parser = PydanticOutputParser(pydantic_object=Person)

print("\n4.1. Инструкции формата:")
print(parser.get_format_instructions()[:200] + "...")

model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)

prompt = ChatPromptTemplate.from_template(
    """Создай вымышленного персонажа.
    {format_instructions}"""
)

chain = prompt.partial(format_instructions=parser.get_format_instructions()) | model | parser

print("\n4.2. Результат парсинга:")
result = chain.invoke({})
print(f"  Тип: {type(result).__name__}")
print(f"  Name: {result.name}")
print(f"  Age: {result.age}")
print(f"  Occupation: {result.occupation}")


ПРИМЕР 4: PydanticOutputParser (базовый)

4.1. Инструкции формата:
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "ty...

4.2. Результат парсинга:
  Тип: Person
  Name: Алексей Смирнов
  Age: 30
  Occupation: Художник


In [ ]:
"""
PydanticOutputParser с вложенными структурами.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 5: PydanticOutputParser (вложенные структуры)")
print("=" * 60)

from langchain_core.output_parsers import PydanticOutputParser


class Address(BaseModel):
    city: str = Field(description="Город")
    country: str = Field(description="Страна")


class Company(BaseModel):
    name: str = Field(description="Название компании")
    industry: str = Field(description="Отрасль")
    employees: int = Field(description="Количество сотрудников")
    headquarters: Address = Field(description="Штаб-квартира")
    technologies: List[str] = Field(description="Используемые технологии")


parser = PydanticOutputParser(pydantic_object=Company)
model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)

prompt = ChatPromptTemplate.from_template(
    """Придумай технологическую компанию.
    {format_instructions}"""
)

chain = prompt.partial(format_instructions=parser.get_format_instructions()) | model | parser

print("\n5.1. Вложенная структура:")
result = chain.invoke({})
print(f"  Company: {result.name}")
print(f"  Industry: {result.industry}")
print(f"  Employees: {result.employees}")
print(f"  HQ: {result.headquarters.city}, {result.headquarters.country}")
print(f"  Technologies: {result.technologies}")


ПРИМЕР 5: PydanticOutputParser (вложенные структуры)

5.1. Вложенная структура:
  Company: TechNova
  Industry: Информационные технологии
  Employees: 250
  HQ: Москва, Россия
  Technologies: ['Искусственный интеллект', 'Большие данные', 'Облачные вычисления', 'Интернет вещей']


# ============================================================================
# ЧАСТЬ 3: WITH_STRUCTURED_OUTPUT
# ============================================================================


In [ ]:
"""
with_structured_output — гарантированный JSON.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 6: with_structured_output (базовый)")
print("=" * 60)


class MovieReview(BaseModel):
    title: str = Field(description="Название фильма")
    rating: float = Field(description="Оценка от 1 до 10")
    summary: str = Field(description="Краткое описание")
    recommend: bool = Field(description="Рекомендуете ли вы фильм")


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)
structured_model = model.with_structured_output(MovieReview)

prompt = ChatPromptTemplate.from_template("Напиши рецензию на фильм: {movie}")

chain = prompt | structured_model

print("\n6.1. Результат с гарантированной структурой:")
result = chain.invoke({"movie": "Интерстеллар"})
print(f"  Title: {result.title}")
print(f"  Rating: {result.rating}")
print(f"  Summary: {result.summary[:80]}...")
print(f"  Recommend: {result.recommend}")


ПРИМЕР 6: with_structured_output (базовый)

6.1. Результат с гарантированной структурой:
  Title: Интерстеллар
  Rating: 9.0
  Summary: Фильм "Интерстеллар" — это эпическая научно-фантастическая драма, рассказывающая...
  Recommend: True


In [ ]:
"""
Сравнение методов: json_mode vs function_calling.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 7: Методы structured output")
print("=" * 60)


class SimpleData(BaseModel):
    name: str
    value: int


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)

print("\n7.1. method='json_mode':")
print("  - Модель генерирует JSON напрямую")
print("  - Быстрее для простых схем")

print("\n7.2. method='function_calling':")
print("  - Модель 'вызывает функцию' со схемой")
print("  - Надёжнее для сложных схем")

# По умолчанию LangChain выбирает оптимальный метод
structured = model.with_structured_output(SimpleData)
result = structured.invoke("Придумай название и число")
print(f"\n7.3. Результат: name='{result.name}', value={result.value}")



ПРИМЕР 7: Методы structured output

7.1. method='json_mode':
  - Модель генерирует JSON напрямую
  - Быстрее для простых схем

7.2. method='function_calling':
  - Модель 'вызывает функцию' со схемой
  - Надёжнее для сложных схем

7.3. Результат: name='Загадочный остров', value=42


In [ ]:
"""
Strict mode для абсолютного соответствия схеме.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 8: Strict mode")
print("=" * 60)


class Product(BaseModel):
    name: str
    price: float
    in_stock: bool


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)

# strict=True гарантирует точное соответствие схеме
structured = model.with_structured_output(Product, strict=True)

print("\n8.1. С strict=True:")
print("  - Гарантированное соответствие схеме")
print("  - Не все функции Pydantic поддерживаются")

result = structured.invoke("Опиши товар: ноутбук")
print(f"\n8.2. Результат: {result}")


ПРИМЕР 8: Strict mode

8.1. С strict=True:
  - Гарантированное соответствие схеме
  - Не все функции Pydantic поддерживаются

8.2. Результат: name='Ноутбук ASUS ZenBook 14' price=749.99 in_stock=True



# ============================================================================
# ЧАСТЬ 4: ОБРАБОТКА ОШИБОК ПАРСИНГА
# ============================================================================


In [ ]:
"""
Типичные ошибки парсинга и их обработка.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 9: Ошибки парсинга")
print("=" * 60)

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.exceptions import OutputParserException


class StrictSchema(BaseModel):
    number: int
    name: str


parser = PydanticOutputParser(pydantic_object=StrictSchema)

# Пробуем распарсить невалидный JSON
invalid_inputs = [
    '{"number": "not a number", "name": "test"}',  # Неверный тип
    '{"number": 42}',  # Отсутствует поле
    "This is not JSON at all",  # Не JSON
]

print("\n9.1. Попытки парсинга невалидных данных:")
for invalid in invalid_inputs:
    try:
        result = parser.parse(invalid)
        print(f"  '{invalid[:30]}...' -> OK")
    except OutputParserException as e:
        print(f"  '{invalid[:30]}...' -> ERROR: {str(e)[:50]}...")
    except Exception as e:
        print(f"  '{invalid[:30]}...' -> ERROR: {type(e).__name__}")



ПРИМЕР 9: Ошибки парсинга

9.1. Попытки парсинга невалидных данных:
  '{"number": "not a number", "na...' -> ERROR: Failed to parse StrictSchema from completion {"num...
  '{"number": 42}...' -> ERROR: Failed to parse StrictSchema from completion {"num...
  'This is not JSON at all...' -> ERROR: Invalid json output: This is not JSON at all
For t...


In [42]:
"""
LEGACY:
OutputFixingParser — автоматическое исправление ошибок.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 10: OutputFixingParser")
print("=" * 60)

from langchain_core.output_parsers import PydanticOutputParser
from langchain.output_parsers.fix import OutputFixingParser


class DataSchema(BaseModel):
    name: str
    count: int


base_parser = PydanticOutputParser(pydantic_object=DataSchema)
model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)

# OutputFixingParser использует LLM для исправления ошибок
fixing_parser = OutputFixingParser.from_llm(parser=base_parser, llm=model)

print("\n10.1. Исправление невалидного JSON:")
# Этот JSON имеет ошибку в типе
broken_json = '{"name": "test", "count": "пять"}'

try:
    # base_parser упадёт
    base_parser.parse(broken_json)
except Exception as e:
    print(f"  base_parser: ERROR - {type(e).__name__}")

# fixing_parser попробует исправить
try:
    result = fixing_parser.parse(broken_json)
    print(f"  fixing_parser: OK - {result}")
except Exception as e:
    print(f"  fixing_parser: ERROR - {e}")


ПРИМЕР 10: OutputFixingParser


ModuleNotFoundError: No module named 'langchain.output_parsers'

In [ ]:
"""
RetryWithErrorOutputParser — повторная попытка с контекстом.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 11: RetryWithErrorOutputParser (концептуально)")
print("=" * 60)

print("""
from langchain.output_parsers import RetryWithErrorOutputParser

retry_parser = RetryWithErrorOutputParser.from_llm(
    parser=base_parser,
    llm=model
)

# При ошибке отправляет модели:
# "Твой предыдущий ответ не прошёл валидацию.
# Ошибка: ...
# Исходный промпт: ...
# Исправь ответ:"

# Отличие от OutputFixingParser:
# - RetryParser имеет доступ к исходному промпту
# - Может лучше понять контекст для исправления
""")


ПРИМЕР 11: RetryWithErrorOutputParser (концептуально)

from langchain.output_parsers import RetryWithErrorOutputParser

retry_parser = RetryWithErrorOutputParser.from_llm(
    parser=base_parser,
    llm=model
)

# При ошибке отправляет модели:
# "Твой предыдущий ответ не прошёл валидацию.
# Ошибка: ...
# Исходный промпт: ...
# Исправь ответ:"

# Отличие от OutputFixingParser:
# - RetryParser имеет доступ к исходному промпту
# - Может лучше понять контекст для исправления



In [ ]:
"""
Стратегия fallback при ошибках парсинга.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 12: Fallback стратегия")
print("=" * 60)

from langchain_core.runnables import chain


class Result(BaseModel):
    answer: str
    confidence: float


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)


@chain
def safe_structured_call(input_data: Dict) -> Dict:
    """Безопасный вызов с fallback."""
    try:
        structured = model.with_structured_output(Result, strict=True)
        prompt = ChatPromptTemplate.from_template("Ответь на вопрос: {question}")
        result = (prompt | structured).invoke(input_data)
        return {"success": True, "data": {"answer": result.answer, "confidence": result.confidence}}
    except Exception as e:
        # Fallback: просто строковый ответ
        basic_chain = ChatPromptTemplate.from_template("Ответь кратко: {question}") | model | StrOutputParser()
        answer = basic_chain.invoke(input_data)
        return {"success": False, "data": {"answer": answer, "confidence": 0.0}, "error": str(e)}


print("\n12.1. Безопасный вызов:")
result = safe_structured_call.invoke({"question": "Что такое Python?"})
print(f"  Success: {result['success']}")
print(f"  Answer: {result['data']['answer'][:60]}...")
print(f"  Confidence: {result['data']['confidence']}")



ПРИМЕР 12: Fallback стратегия

12.1. Безопасный вызов:
  Success: True
  Answer: Python — это высокоуровневый язык программирования, который ...
  Confidence: 0.95


# ============================================================================
# ЧАСТЬ 5: ПРОДВИНУТЫЕ ПАТТЕРНЫ
# ============================================================================


In [ ]:
"""
Использование Enum для ограниченных значений.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 13: Enum типы")
print("=" * 60)


class Sentiment(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"


class SentimentAnalysis(BaseModel):
    text: str = Field(description="Анализируемый текст")
    sentiment: Sentiment = Field(description="Тональность")
    confidence: float = Field(description="Уверенность от 0 до 1")


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)
structured = model.with_structured_output(SentimentAnalysis)

prompt = ChatPromptTemplate.from_template("Определи тональность текста: {text}")

chain = prompt | structured

texts = [
    "Это лучший продукт, который я когда-либо покупал!",
    "Ужасное качество, полное разочарование.",
    "Товар соответствует описанию.",
]

print("\n13.1. Классификация с Enum:")
for text in texts:
    result = chain.invoke({"text": text})
    print(f"  '{text[:40]}...'")
    print(f"    -> {result.sentiment.value} (confidence: {result.confidence:.2f})")



ПРИМЕР 13: Enum типы

13.1. Классификация с Enum:
  'Это лучший продукт, который я когда-либо...'
    -> positive (confidence: 0.95)
  'Ужасное качество, полное разочарование....'
    -> negative (confidence: 0.95)
  'Товар соответствует описанию....'
    -> positive (confidence: 0.95)


In [ ]:
"""
Optional и Union типы в схемах.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 14: Optional и Union типы")
print("=" * 60)


class SearchResult(BaseModel):
    query: str
    found: bool
    result: Optional[str] = Field(default=None, description="Результат если найден")
    alternatives: Optional[List[str]] = Field(default=None, description="Альтернативы если не найдено")


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)
structured = model.with_structured_output(SearchResult)

prompt = ChatPromptTemplate.from_template(
    "Поищи информацию о: {query}. Если не нашёл точный ответ, предложи альтернативы."
)

chain = prompt | structured

print("\n14.1. Поиск с optional полями:")
result = chain.invoke({"query": "столица Франции"})
print(f"  Query: {result.query}")
print(f"  Found: {result.found}")
print(f"  Result: {result.result}")
print(f"  Alternatives: {result.alternatives}")



ПРИМЕР 14: Optional и Union типы

14.1. Поиск с optional полями:
  Query: столица Франции
  Found: True
  Result: Париж
  Alternatives: None


In [ ]:
"""
Кастомные валидаторы Pydantic.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 15: Кастомные валидаторы")
print("=" * 60)

from pydantic import field_validator


class Review(BaseModel):
    title: str
    rating: float
    text: str

    @field_validator("rating")
    @classmethod
    def rating_in_range(cls, v):
        if not 1 <= v <= 10:
            raise ValueError("Rating must be between 1 and 10")
        return v

    @field_validator("text")
    @classmethod
    def text_not_empty(cls, v):
        if len(v.strip()) < 10:
            raise ValueError("Review text must be at least 10 characters")
        return v


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)
structured = model.with_structured_output(Review)

prompt = ChatPromptTemplate.from_template("Напиши рецензию на книгу: {book}")

chain = prompt | structured

print("\n15.1. Модель с кастомными валидаторами:")
result = chain.invoke({"book": "Война и мир"})
print(f"  Title: {result.title}")
print(f"  Rating: {result.rating}")
print(f"  Text: {result.text[:80]}...")



ПРИМЕР 15: Кастомные валидаторы

15.1. Модель с кастомными валидаторами:
  Title: Рецензия на книгу "Война и мир"
  Rating: 5.0
  Text: "Война и мир" Льва Толстого — это не просто роман, а целая эпоха, запечатленная ...


# ============================================================================
# ЧАСТЬ 6: ПРАКТИЧЕСКИЕ РЕЦЕПТЫ
# ============================================================================


In [ ]:
"""
Извлечение сущностей из текста.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 16: Извлечение сущностей")
print("=" * 60)


class Entity(BaseModel):
    text: str = Field(description="Текст сущности")
    type: str = Field(description="Тип: PERSON, ORGANIZATION, LOCATION, DATE")
    start: int = Field(description="Начальная позиция в тексте")
    end: int = Field(description="Конечная позиция в тексте")


class ExtractionResult(BaseModel):
    entities: List[Entity] = Field(description="Найденные сущности")


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)
structured = model.with_structured_output(ExtractionResult)

prompt = ChatPromptTemplate.from_template(
    """Извлеки именованные сущности из текста.
    Типы: PERSON, ORGANIZATION, LOCATION, DATE.

    Текст: {text}"""
)

chain = prompt | structured

text = "Илон Маск основал SpaceX в 2002 году в Калифорнии."

print(f"\n16.1. Текст: '{text}'")
result = chain.invoke({"text": text})

print("\n16.2. Найденные сущности:")
for entity in result.entities:
    print(f"  '{entity.text}' ({entity.type}) [{entity.start}:{entity.end}]")



ПРИМЕР 16: Извлечение сущностей

16.1. Текст: 'Илон Маск основал SpaceX в 2002 году в Калифорнии.'

16.2. Найденные сущности:
  'Илон Маск' (PERSON) [0:10]
  'SpaceX' (ORGANIZATION) [17:23]
  '2002' (DATE) [27:31]
  'Калифорнии' (LOCATION) [35:46]


In [ ]:
"""
Классификация с объяснением.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 17: Классификация с объяснением")
print("=" * 60)


class Category(str, Enum):
    TECHNICAL = "technical"
    BILLING = "billing"
    GENERAL = "general"
    COMPLAINT = "complaint"


class ClassificationResult(BaseModel):
    category: Category = Field(description="Категория обращения")
    confidence: float = Field(description="Уверенность от 0 до 1")
    reasoning: str = Field(description="Объяснение классификации")
    keywords: List[str] = Field(description="Ключевые слова")


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)
structured = model.with_structured_output(ClassificationResult)

prompt = ChatPromptTemplate.from_template(
    """Классифицируй обращение клиента.

    Категории:
    - technical: технические проблемы
    - billing: вопросы оплаты
    - general: общие вопросы
    - complaint: жалобы

    Обращение: {message}"""
)

chain = prompt | structured

messages = [
    "Приложение вылетает при попытке загрузить фото",
    "Почему с меня списали деньги дважды?",
    "Это уже третий раз когда доставка опаздывает!",
]

print("\n17.1. Классификация обращений:")
for msg in messages:
    result = chain.invoke({"message": msg})
    print(f"\n  Обращение: '{msg[:40]}...'")
    print(f"  Категория: {result.category.value}")
    print(f"  Уверенность: {result.confidence:.2f}")
    print(f"  Причина: {result.reasoning}")
    print(f"  Ключевые слова: {result.keywords}")



ПРИМЕР 17: Классификация с объяснением

17.1. Классификация обращений:

  Обращение: 'Приложение вылетает при попытке загрузит...'
  Категория: technical
  Уверенность: 0.95
  Причина: Обращение связано с технической проблемой, так как клиент сообщает о вылете приложения при попытке загрузки фото.
  Ключевые слова: ['приложение', 'вылетает', 'загрузка', 'фото', 'техническая проблема']

  Обращение: 'Почему с меня списали деньги дважды?...'
  Категория: billing
  Уверенность: 0.95
  Причина: Обращение связано с вопросом о списании денег, что указывает на проблему с оплатой. Это соответствует категории 'billing'.
  Ключевые слова: ['списание', 'деньги', 'дважды', 'вопросы оплаты']

  Обращение: 'Это уже третий раз когда доставка опазды...'
  Категория: complaint
  Уверенность: 0.95
  Причина: Клиент выражает недовольство по поводу повторяющейся проблемы с доставкой, что указывает на жалобу.
  Ключевые слова: ['доставка', 'опоздание', 'жалоба', 'проблема']


In [ ]:
"""
Пошаговый анализ (Chain of Thought).
"""
print("\n" + "=" * 60)
print("ПРИМЕР 18: Пошаговый анализ")
print("=" * 60)


class Step(BaseModel):
    step_number: int = Field(description="Номер шага")
    action: str = Field(description="Действие")
    result: str = Field(description="Результат")


class Analysis(BaseModel):
    steps: List[Step] = Field(description="Шаги рассуждения")
    final_answer: str = Field(description="Финальный ответ")
    confidence: float = Field(description="Уверенность в ответе")


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)
structured = model.with_structured_output(Analysis)

prompt = ChatPromptTemplate.from_template(
    """Реши задачу пошагово.

    Задача: {problem}"""
)

chain = prompt | structured

problem = "В магазине было 50 яблок. Утром продали 1/5 всех яблок, а вечером ещё 15. Сколько яблок осталось?"

print(f"\n18.1. Задача: {problem}")
result = chain.invoke({"problem": problem})

print("\n18.2. Решение:")
for step in result.steps:
    print(f"  Шаг {step.step_number}: {step.action}")
    print(f"    Результат: {step.result}")

print(f"\n18.3. Ответ: {result.final_answer}")
print(f"  Уверенность: {result.confidence:.2f}")



ПРИМЕР 18: Пошаговый анализ

18.1. Задача: В магазине было 50 яблок. Утром продали 1/5 всех яблок, а вечером ещё 15. Сколько яблок осталось?

18.2. Решение:
  Шаг 1: Определить количество яблок, проданных утром.
    Результат: Утром продали 1/5 от 50 яблок, что составляет 50 / 5 = 10 яблок.
  Шаг 2: Вычесть количество проданных яблок утром из общего количества.
    Результат: После утренней продажи осталось 50 - 10 = 40 яблок.
  Шаг 3: Определить количество яблок, проданных вечером.
    Результат: Вечером продали 15 яблок.
  Шаг 4: Вычесть количество проданных яблок вечером из оставшегося количества.
    Результат: После вечерней продажи осталось 40 - 15 = 25 яблок.

18.3. Ответ: Осталось 25 яблок.
  Уверенность: 0.95


In [ ]:
"""
Генерация нескольких вариантов.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 19: Множественные варианты")
print("=" * 60)


class Variant(BaseModel):
    text: str = Field(description="Текст варианта")
    style: str = Field(description="Стиль: formal, casual, creative")


class MultipleVariants(BaseModel):
    variants: List[Variant] = Field(description="Варианты текста", min_length=3)


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0.7,
)
structured = model.with_structured_output(MultipleVariants)

prompt = ChatPromptTemplate.from_template(
    """Создай 3 варианта сообщения для {purpose}.
    Используй разные стили: formal, casual, creative."""
)

chain = prompt | structured

print("\n19.1. Варианты приветствия:")
result = chain.invoke({"purpose": "приветствие клиента"})

for i, variant in enumerate(result.variants, 1):
    print(f"\n  Вариант {i} ({variant.style}):")
    print(f"    {variant.text}")


ПРИМЕР 19: Множественные варианты

19.1. Варианты приветствия:

  Вариант 1 (formal):
    Уважаемый клиент,

Благодарим вас за обращение в нашу компанию. Мы рады приветствовать вас и готовы оказать вам всю необходимую помощь.

С уважением,
Команда поддержки

  Вариант 2 (casual):
    Привет! 👋

Рады видеть тебя у нас! Если у тебя есть вопросы или нужна помощь, просто дай знать — мы здесь, чтобы помочь!

  Вариант 3 (creative):
    Здравствуйте, наш дорогой клиент! 🌟

Мы счастливы, что вы с нами! Погрузитесь в мир удивительных возможностей и не стесняйтесь задавать вопросы. Давайте создавать что-то великолепное вместе!


# ============================================================================
# ЧАСТЬ 7: ИНТЕГРАЦИЯ С LCEL
# ============================================================================


In [ ]:
"""
Парсер как часть LCEL цепочки.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 20: Парсер в LCEL цепочке")
print("=" * 60)

from langchain_core.runnables import RunnablePassthrough


class ExtractedData(BaseModel):
    main_topic: str
    keywords: List[str]
    summary: str


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0,
)

# Цепочка: обогащение -> анализ -> парсинг
chain = (
    RunnablePassthrough.assign(word_count=lambda x: len(x["text"].split()))
    | ChatPromptTemplate.from_template(
        """Проанализируй текст ({word_count} слов):
        {text}

        Выдели главную тему, ключевые слова и напиши краткое резюме."""
    )
    | model.with_structured_output(ExtractedData)
)

text = """
Искусственный интеллект стремительно развивается.
Машинное обучение и нейронные сети находят применение
в медицине, транспорте и финансах. Эксперты прогнозируют
дальнейший рост этой технологии.
"""

print("\n20.1. Анализ текста:")
result = chain.invoke({"text": text})
print(f"  Тема: {result.main_topic}")
print(f"  Ключевые слова: {result.keywords}")
print(f"  Резюме: {result.summary}")



ПРИМЕР 20: Парсер в LCEL цепочке

20.1. Анализ текста:
  Тема: Развитие искусственного интеллекта
  Ключевые слова: ['искусственный интеллект', 'машинное обучение', 'нейронные сети', 'медицина', 'транспорт', 'финансы', 'прогнозы', 'технологии']
  Резюме: Текст описывает стремительное развитие искусственного интеллекта, его применение в различных сферах, таких как медицина, транспорт и финансы, а также прогнозирует дальнейший рост этой технологии.


In [ ]:
"""
Условный выбор парсера.
"""
print("\n" + "=" * 60)
print("ПРИМЕР 21: Условный парсинг")
print("=" * 60)

from langchain_core.runnables import RunnableBranch


class Question(BaseModel):
    question_text: str
    category: str


class Command(BaseModel):
    action: str
    parameters: Dict[str, str]


model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    model="google/gemini-3-flash-preview",
    temperature=0,
)

# Сначала определяем тип
classifier = (
    ChatPromptTemplate.from_template("Это вопрос или команда? Ответь одним словом: question или command\n\n{input}")
    | model
    | StrOutputParser()
)

# Разные парсеры для разных типов
question_chain = model.with_structured_output(Question)
command_chain = model.with_structured_output(Command)


def route(x):
    input_type = x["type"].strip().lower()
    if "question" in input_type:
        return question_chain.invoke(f"Разбери вопрос: {x['input']}")
    else:
        return command_chain.invoke(f"Разбери команду: {x['input']}")


chain = RunnablePassthrough.assign(type=classifier) | route

print("\n21.1. Вопрос:")
result1 = chain.invoke({"input": "Какая погода в Москве?"})
print(f"  Тип результата: {type(result1).__name__}")
print(f"  Данные: {result1}")

print("\n21.2. Команда:")
result2 = chain.invoke({"input": "Установи напоминание на завтра 10:00"})
print(f"  Тип результата: {type(result2).__name__}")
print(f"  Данные: {result2}")



ПРИМЕР 21: Условный парсинг

21.1. Вопрос:
  Тип результата: Question
  Данные: question_text='Какая погода в Москве?' category='Weather'

21.2. Команда:
  Тип результата: Command
  Данные: action='set_reminder' parameters={'date': 'tomorrow', 'time': '10:00', 'text': 'напоминание'}
